# QC: наличие месячной комиссии — MPOS_RENT vs ЦФТ vs `final_df`

Сравнение **наличия** месячной комиссии в двух источниках и вхождения этих договоров/клиентов в уже собранный `final_df`.

| Источник | Система | Озёрная таблица |
|---|---|---|
| **MPOS_RENT** | Альфа | `ods_alpha.scd1_mrc_pos_rent` (`n_amt`, `d_rent`, `c_nmrc`) |
| **OPER_DOCUM / DOG_OPER** | ЦФТ | `ods.scd1_z_R2_IP_DOG_OPER` + `VID_COMISS` + merchants + client |

В озере отдельной `OPER_DOCUM` для эквайринговых комиссий нет: факт месячных комиссий ЦФТ лежит в `R2_IP_DOG_OPER` (операции по договору ТСП). Тетрадка сначала ищет таблицы `*oper_docum*`, затем считает покрытие по `DOG_OPER`.

## Что считается
- **Наличие в источнике** = есть строка за месяц с суммой (в т.ч. 0; ненулевые считаются отдельно).
- **Вхождение в `final_df`** = ключ есть в витрине за тот же месяц.
- Зёрна: `agr_id` (договор) и `inn` (клиент).
- Окно: последние **3 месяца** из `final_df` (можно задать вручную).

## Как запускать
В **том же kernel**, где уже есть `final_df` / `final_df_period_df` и желательно `imp`.


In [ ]:
import re
from decimal import Decimal, InvalidOperation
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 120)
pd.set_option('display.width', 220)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}'.replace(',', ' '))

DATA_DIR = Path('/home/jovyan/documents/Equaring/Data')
OUT_DIR = DATA_DIR / 'qc_mpos_vs_cft_oper_docum_3m'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# None = взять последние 3 месяца из final_df
FOCUS_MONTHS = None  # например: ['2026-06', '2026-07', '2026-08']

# ЦФТ: сначала все виды комиссий, затем подмножество «месячных»
CFT_MONTHLY_TYPE_NEEDLES = (
    'месяч', 'фикс', 'аренд', 'абонент',
)
# True: покрытие ЦФТ считать по отфильтрованным видам; False — по всем DOG_OPER
CFT_FILTER_MONTHLY_TYPES = True

MEM_LIMIT = '8g'
print('OUT_DIR:', OUT_DIR)
print('FOCUS_MONTHS override:', FOCUS_MONTHS)


## 0) Helpers + `final_df` из памяти


In [ ]:
def normalize_inn_q1(v):
    if pd.isna(v):
        return None
    s = str(v).strip()
    s = re.sub(r'\.0$', '', s)
    s = re.sub(r'\D+', '', s)
    if not s:
        return None
    if len(s) == 9:
        s = s.zfill(10)
    elif len(s) == 11:
        s = s.zfill(12)
    return s if len(s) in (10, 12) else None


def normalize_agr_q1(v):
    if pd.isna(v):
        return None
    s = str(v).strip().replace('\xa0', '').replace(' ', '').replace(',', '.')
    if s in {'', 'nan', 'None'}:
        return None
    try:
        d = Decimal(s)
        if d == d.to_integral_value():
            return str(int(d))
    except (InvalidOperation, ValueError):
        pass
    s = re.sub(r'\.0$', '', s)
    return s if s not in {'', 'nan', 'None'} else None


def month_bounds(months):
    starts = [pd.Timestamp(f'{m}-01') for m in months]
    period_start = min(starts).strftime('%Y-%m-%d')
    period_end = (max(starts) + pd.offsets.MonthEnd(1)).strftime('%Y-%m-%d')
    period_end_excl = (max(starts) + pd.offsets.MonthBegin(1)).strftime('%Y-%m-%d')
    return period_start, period_end, period_end_excl


def fetch_imp(sql, label):
    t0 = pd.Timestamp.now()
    print(f'FETCH {label} ...')
    with imp:
        imp.execute(f'set MEM_LIMIT={MEM_LIMIT}')
        df = imp.fetch(sql)
    if df is None:
        df = pd.DataFrame()
    dt = (pd.Timestamp.now() - t0).total_seconds()
    print(f'  rows={len(df):,}  {dt:.1f}s')
    return df


def is_monthly_cft_type(name):
    if pd.isna(name):
        return False
    s = str(name).lower()
    return any(n in s for n in CFT_MONTHLY_TYPE_NEEDLES)


def coverage_block(src_keys, fin_keys):
    src = {k for k in src_keys if k}
    fin = {k for k in fin_keys if k}
    inter = src & fin
    n_src, n_fin, n_both = len(src), len(fin), len(inter)
    return {
        'n_source': n_src,
        'n_final': n_fin,
        'n_source_in_final': n_both,
        'pct_source_in_final': round(100.0 * n_both / n_src, 2) if n_src else np.nan,
        'n_final_in_source': n_both,
        'pct_final_has_source': round(100.0 * n_both / n_fin, 2) if n_fin else np.nan,
        'n_source_only': n_src - n_both,
        'n_final_only': n_fin - n_both,
    }


def resolve_final_df():
    candidates = []
    for name in ['final_df_period_df', 'final_df', 'final_df_period']:
        obj = globals().get(name)
        if isinstance(obj, pd.DataFrame) and len(obj):
            candidates.append((name, obj))
    by_month = globals().get('final_df_by_month')
    if isinstance(by_month, dict):
        parts = [df for df in by_month.values() if isinstance(df, pd.DataFrame) and len(df)]
        if parts:
            candidates.append(('final_df_by_month(concat)', pd.concat(parts, ignore_index=True)))
    if not candidates:
        raise RuntimeError(
            'final_df не найден в памяти kernel. '
            'Запусти эту тетрадку в том же kernel, где уже собран final_df / final_df_period_df.'
        )
    name, df = candidates[0]
    print(f'OK: in-memory `{name}` rows={len(df):,}')
    return df.copy()


fd_raw = resolve_final_df()
need_cols = ['report_month', 'inn', 'agr_id']
missing = [c for c in need_cols if c not in fd_raw.columns]
if missing:
    raise RuntimeError(f'final_df без колонок: {missing}')

fd = fd_raw.copy()
fd['report_month'] = (
    pd.to_datetime(fd['report_month'], errors='coerce').dt.strftime('%Y-%m')
)
fd['inn_key'] = fd['inn'].map(normalize_inn_q1)
fd['agr_id_key'] = fd['agr_id'].map(normalize_agr_q1)
if 'commission_monthly' in fd.columns:
    fd['commission_monthly_num'] = pd.to_numeric(fd['commission_monthly'], errors='coerce')
else:
    fd['commission_monthly_num'] = np.nan

months_all = sorted(fd['report_month'].dropna().unique().tolist())
if FOCUS_MONTHS:
    months = [m for m in FOCUS_MONTHS if m in set(months_all)]
    if len(months) != len(FOCUS_MONTHS):
        print('WARN: запрошенные месяцы отсутствуют в final_df:', set(FOCUS_MONTHS) - set(months))
else:
    months = months_all[-3:]

if len(months) < 1:
    raise RuntimeError(f'Нет месяцев для сравнения. final_df months={months_all}')

period_start, period_end, period_end_excl = month_bounds(months)
fd3 = fd.loc[fd['report_month'].isin(months)].copy()

print('final_df months all:', months_all)
print('FOCUS 3 months:', months)
print(f'period: {period_start} .. {period_end} (end excl {period_end_excl})')
print(f'final_df rows in window: {len(fd3):,} | agr={fd3["agr_id_key"].nunique():,} | inn={fd3["inn_key"].nunique():,}')
display(
    fd3.groupby('report_month', as_index=False)
    .agg(
        rows=('agr_id_key', 'size'),
        agr=('agr_id_key', 'nunique'),
        inn=('inn_key', 'nunique'),
        commission_monthly_sum=('commission_monthly_num', 'sum'),
        commission_monthly_nonzero=('commission_monthly_num', lambda s: int(pd.to_numeric(s, errors='coerce').fillna(0).ne(0).sum())),
    )
    .sort_values('report_month')
)


## 1) Impala: reuse `imp` + поиск `OPER_DOCUM`


In [ ]:
from rail_connectors.connection import connect

if 'imp' in globals() and imp is not None:
    print('Reuse existing imp connection')
else:
    imp = connect(
        to='IMPALA',
        extra_options={'db': 'sandbox_ai'},
        driver_args={'tez.queue.name': 'ai'},
        kerberos={
            'keytab_path': '/home/jovyan/test_requests/tech.keytab',
            'use_credentials': True,
            'update_keytab': True,
        },
        user_params={'user_name': 'Shestopalov-VYur'},
    )
    imp._init_connection()
    print('Impala connected')


In [ ]:
def list_tables_like(schema, needle):
    sql = f"show tables in {schema} like '*{needle}*'"
    try:
        df = fetch_imp(sql, f'show tables {schema} *{needle}*')
    except Exception as exc:
        print(f'  skip {schema}: {type(exc).__name__}: {exc}')
        return pd.DataFrame()
    if df is None or df.empty:
        return pd.DataFrame()
    col = df.columns[0]
    out = df[[col]].copy()
    out.columns = ['table_name']
    out['schema'] = schema
    out['fqn'] = out['schema'] + '.' + out['table_name'].astype(str)
    return out


probe_schemas = ['ods', 'ods_cft', 'ods_alpha', 'sandbox_ai']
needles = ['oper_docum', 'OPER_DOCUM', 'dog_oper', 'DOG_OPER']
found_tables = []
for schema in probe_schemas:
    for needle in needles:
        part = list_tables_like(schema, needle)
        if len(part):
            found_tables.append(part)

cft_catalog = (
    pd.concat(found_tables, ignore_index=True).drop_duplicates('fqn')
    if found_tables else pd.DataFrame(columns=['table_name', 'schema', 'fqn'])
)
print('=== таблицы *oper_docum* / *dog_oper* ===')
display(cft_catalog)

CFT_DOG_OPER_CANDIDATES = [
    'ods.scd1_z_R2_IP_DOG_OPER',
    'ods.scd1_z_r2_ip_dog_oper',
]
CFT_VID_CANDIDATES = [
    'ods.scd1_z_R2_VID_COMISS',
    'ods.scd1_z_r2_vid_comiss',
]
CFT_MERCH_CANDIDATES = [
    'ods.scd1_z_r2_ip_merchants',
    'ods.scd1_z_R2_IP_MERCHANTS',
]
CFT_CLIENT = 'ods.scd1_z_client'

oper_docum_hits = cft_catalog.loc[
    cft_catalog['table_name'].astype(str).str.lower().str.contains('oper_docum', na=False)
]
print('OPER_DOCUM hits:', oper_docum_hits['fqn'].tolist() if len(oper_docum_hits) else 'нет')
if len(oper_docum_hits):
    sample_fqn = oper_docum_hits['fqn'].iloc[0]
    try:
        desc = fetch_imp(f'describe {sample_fqn}', f'describe {sample_fqn}')
        print(f'describe {sample_fqn}')
        display(desc)
    except Exception as exc:
        print('describe failed:', type(exc).__name__, exc)


## 2) MPOS_RENT (Альфа)

Тот же маппинг, что в `final_script_2` шаг 10m: `c_nmrc → agr_terms → agreements(SA) → companies → inn+agr_id`.
Для покрытия берём **все** успешно смапленные ключи (не только unique `c_nmrc`).


In [ ]:
sql_mpos = f'''
with rent_base as (
  select
    cast(c_nmrc as string) as c_nmrc,
    cast(d_rent as date) as d_rent_dt,
    cast(n_amt as double) as n_amt_num,
    cast(ods_commit_ts as timestamp) as ods_commit_ts,
    cast(ods_insert_ts as timestamp) as ods_insert_ts,
    cast(ods_op_csn as decimal(38, 0)) as ods_op_csn,
    coalesce(cast(ods_deleted_flg as string), '0') as ods_deleted_flg
  from ods_alpha.scd1_mrc_pos_rent
  where c_nmrc is not null
    and cast(d_rent as date) between cast('{period_start}' as date) and cast('{period_end}' as date)
),
rent_ranked as (
  select
    *,
    row_number() over (
      partition by c_nmrc, d_rent_dt
      order by coalesce(ods_commit_ts, ods_insert_ts) desc, ods_op_csn desc, ods_insert_ts desc
    ) as rn
  from rent_base
  where ods_deleted_flg not in ('1', 'Y', 'y')
),
rent_dedup as (
  select c_nmrc, d_rent_dt, n_amt_num
  from rent_ranked
  where rn = 1
),
terms_active as (
  select distinct
    cast(t.n_agr as string) as n_agr,
    cast(t.c_nmrc as string) as c_nmrc,
    cast(t.d_valid_from as date) as d_valid_from,
    cast(t.d_valid_to as date) as d_valid_to
  from ods_alpha.scd1_agr_terms t
  where coalesce(cast(t.ods_deleted_flg as string), '0') not in ('1', 'Y', 'y')
    and t.c_nmrc is not null
    and cast(t.d_valid_from as date) <= cast('{period_end}' as date)
    and (t.d_valid_to is null or cast(t.d_valid_to as date) >= cast('{period_start}' as date))
),
agreements_active as (
  select distinct
    cast(a.n_agr as string) as n_agr,
    cast(a.abs_agr_id as string) as agr_id,
    cast(a.n_cmp_client as string) as n_cmp_client,
    cast(a.d_valid_from as date) as d_valid_from,
    cast(a.d_valid_to as date) as d_valid_to
  from ods_alpha.scd1_agreements a
  where coalesce(cast(a.ods_deleted_flg as string), '0') not in ('1', 'Y', 'y')
    and upper(trim(cast(a.acq_class as string))) = 'SA'
    and cast(a.d_valid_from as date) <= cast('{period_end}' as date)
    and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('{period_start}' as date))
),
companies_active as (
  select distinct
    cast(c.n_cmp as string) as n_cmp,
    regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', '') as inn_key
  from ods_alpha.scd1_companies c
  where coalesce(cast(c.ods_deleted_flg as string), '0') not in ('1', 'Y', 'y')
    and c.c_inn is not null
),
mapped_raw as (
  select
    r.c_nmrc,
    r.d_rent_dt,
    r.n_amt_num,
    c.inn_key,
    cast(a.agr_id as string) as agr_id_key
  from rent_dedup r
  left join terms_active t
    on t.c_nmrc = r.c_nmrc
   and r.d_rent_dt between t.d_valid_from and coalesce(t.d_valid_to, cast('2999-12-31' as date))
  left join agreements_active a
    on a.n_agr = t.n_agr
   and r.d_rent_dt between a.d_valid_from and coalesce(a.d_valid_to, cast('2999-12-31' as date))
  left join companies_active c
    on c.n_cmp = a.n_cmp_client
)
select
  substr(cast(d_rent_dt as string), 1, 7) as report_month,
  inn_key as inn,
  agr_id_key as agr_id,
  count(*) as rent_rows,
  count(distinct c_nmrc) as n_nmrc,
  sum(n_amt_num) as commission_mpos,
  sum(case when n_amt_num is not null then 1 else 0 end) as n_amt_not_null,
  sum(case when n_amt_num is not null and n_amt_num <> 0 then 1 else 0 end) as n_amt_nonzero
from mapped_raw
group by 1, 2, 3
'''

mpos_raw = fetch_imp(sql_mpos, 'MPOS_RENT mapped 3m')
if mpos_raw.empty:
    mpos_raw = pd.DataFrame(columns=[
        'report_month', 'inn', 'agr_id', 'rent_rows', 'n_nmrc',
        'commission_mpos', 'n_amt_not_null', 'n_amt_nonzero',
    ])

mpos = mpos_raw.copy()
mpos['report_month'] = mpos['report_month'].astype(str).str[:7]
mpos['inn_key'] = mpos['inn'].map(normalize_inn_q1)
mpos['agr_id_key'] = mpos['agr_id'].map(normalize_agr_q1)
for c in ['rent_rows', 'n_nmrc', 'commission_mpos', 'n_amt_not_null', 'n_amt_nonzero']:
    mpos[c] = pd.to_numeric(mpos[c], errors='coerce')

mpos_unmapped = mpos.loc[mpos['agr_id_key'].isna()].copy()
mpos_map = (
    mpos.dropna(subset=['agr_id_key'])
    .groupby(['report_month', 'inn_key', 'agr_id_key'], as_index=False)
    .agg(
        rent_rows=('rent_rows', 'sum'),
        n_nmrc=('n_nmrc', 'sum'),
        commission_mpos=('commission_mpos', 'sum'),
        n_amt_not_null=('n_amt_not_null', 'sum'),
        n_amt_nonzero=('n_amt_nonzero', 'sum'),
    )
)
mpos_map['has_mpos'] = True
mpos_map['has_mpos_nonzero'] = mpos_map['commission_mpos'].fillna(0).ne(0)

print('MPOS raw groups:', f'{len(mpos_raw):,}')
print('MPOS mapped agr keys:', f'{len(mpos_map):,}')
print('MPOS groups without agr_id (unmapped / null):', f'{len(mpos_unmapped):,}')
display(
    mpos_map.groupby('report_month', as_index=False)
    .agg(
        agr=('agr_id_key', 'nunique'),
        inn=('inn_key', 'nunique'),
        sum_commission_mpos=('commission_mpos', 'sum'),
        agr_nonzero=('has_mpos_nonzero', 'sum'),
    )
    .sort_values('report_month')
)


## 3) ЦФТ: `DOG_OPER` (OPER_DOCUM-контур)

SQL как у коллег: `DOG_OPER` + `VID_COMISS` + `r2_ip_merchants` + `client`.
Месяц = `c_date_create`. Суммы: `c_calc_summ` / `c_pay_summ`.


In [ ]:
def try_fetch_cft(oper_tbl, vid_tbl, merch_tbl):
    sql = f'''
    select
      regexp_replace(trim(cast(cl.c_inn as string)), '[^0-9]', '') as inn,
      cast(m.id as string) as agr_id,
      cast(m.c_name_in_pr as string) as agreement_num,
      cast(vc.c_name as string) as commis_type,
      cast(o.c_date_create as timestamp) as c_date_create,
      cast(o.c_pay_summ as double) as c_pay_summ,
      cast(o.c_calc_summ as double) as c_calc_summ
    from {oper_tbl} o
    join {vid_tbl} vc on vc.id = o.c_vid_comiss
    join {merch_tbl} m on m.id = o.c_parent_id
    join {CFT_CLIENT} cl on m.c_cl_org = cl.id
    where o.c_parent_class = 'R2_IP_MERCHANTS'
      and cl.class_id = 'CL_ORG'
      and cast(o.c_date_create as date) >= cast('{period_start}' as date)
      and cast(o.c_date_create as date) < cast('{period_end_excl}' as date)
    '''
    return fetch_imp(sql, f'CFT {oper_tbl}')


cft_raw = None
cft_used = None
last_err = None
for oper_tbl in CFT_DOG_OPER_CANDIDATES:
    for vid_tbl in CFT_VID_CANDIDATES:
        for merch_tbl in CFT_MERCH_CANDIDATES:
            try:
                cft_raw = try_fetch_cft(oper_tbl, vid_tbl, merch_tbl)
                cft_used = {'oper': oper_tbl, 'vid': vid_tbl, 'merch': merch_tbl}
                break
            except Exception as exc:
                last_err = exc
                print(f'FAIL {oper_tbl} / {vid_tbl} / {merch_tbl}: {type(exc).__name__}: {exc}')
        if cft_used:
            break
    if cft_used:
        break

if cft_raw is None:
    raise RuntimeError(f'Не удалось прочитать ЦФТ DOG_OPER. last_err={last_err}')

print('CFT tables used:', cft_used)
print('CFT raw rows:', f'{len(cft_raw):,}')

if cft_raw.empty:
    cft_raw = pd.DataFrame(columns=[
        'inn', 'agr_id', 'agreement_num', 'commis_type',
        'c_date_create', 'c_pay_summ', 'c_calc_summ',
    ])

cft = cft_raw.copy()
cft['inn_key'] = cft['inn'].map(normalize_inn_q1)
cft['agr_id_key'] = cft['agr_id'].map(normalize_agr_q1)
cft['c_date_create'] = pd.to_datetime(cft['c_date_create'], errors='coerce')
cft['report_month'] = cft['c_date_create'].dt.strftime('%Y-%m')
cft = cft.loc[cft['report_month'].isin(months)].copy()
cft['c_pay_summ'] = pd.to_numeric(cft['c_pay_summ'], errors='coerce')
cft['c_calc_summ'] = pd.to_numeric(cft['c_calc_summ'], errors='coerce')
cft['is_monthly_type'] = cft['commis_type'].map(is_monthly_cft_type)

print('\\n=== VID_COMISS / commis_type за окно ===')
type_prof = (
    cft.groupby('commis_type', dropna=False)
    .agg(
        rows=('agr_id_key', 'size'),
        agr=('agr_id_key', 'nunique'),
        inn=('inn_key', 'nunique'),
        sum_calc=('c_calc_summ', 'sum'),
        sum_pay=('c_pay_summ', 'sum'),
        monthly_flag=('is_monthly_type', 'max'),
    )
    .sort_values('rows', ascending=False)
    .reset_index()
)
display(type_prof)

cft_all = cft.copy()
cft_month_types = cft.loc[cft['is_monthly_type']].copy()
print(
    f'CFT all types: rows={len(cft_all):,} agr={cft_all["agr_id_key"].nunique():,}'
)
print(
    f'CFT monthly-like types: rows={len(cft_month_types):,} agr={cft_month_types["agr_id_key"].nunique():,}'
)

if CFT_FILTER_MONTHLY_TYPES and len(cft_month_types) == 0:
    print('WARN: фильтр месячных видов дал 0 строк — покрытие ЦФТ считаем по ВСЕМ видам DOG_OPER')
    cft_use = cft_all
    cft_scope = 'all_vid_comiss'
elif CFT_FILTER_MONTHLY_TYPES:
    cft_use = cft_month_types
    cft_scope = 'monthly_like_vid_comiss'
else:
    cft_use = cft_all
    cft_scope = 'all_vid_comiss'

print('CFT coverage scope:', cft_scope)


def agg_cft(src):
    if src.empty:
        return pd.DataFrame(columns=[
            'report_month', 'inn_key', 'agr_id_key', 'cft_rows',
            'commission_cft_calc', 'commission_cft_pay', 'has_cft', 'has_cft_nonzero',
        ])
    out = (
        src.dropna(subset=['agr_id_key'])
        .groupby(['report_month', 'inn_key', 'agr_id_key'], as_index=False)
        .agg(
            cft_rows=('c_date_create', 'size'),
            commission_cft_calc=('c_calc_summ', 'sum'),
            commission_cft_pay=('c_pay_summ', 'sum'),
        )
    )
    out['has_cft'] = True
    out['has_cft_nonzero'] = out['commission_cft_calc'].fillna(0).ne(0) | out['commission_cft_pay'].fillna(0).ne(0)
    return out


cft_map = agg_cft(cft_use)
cft_map_all = agg_cft(cft_all)
print('CFT mapped keys (scope):', f'{len(cft_map):,}')
display(
    cft_map.groupby('report_month', as_index=False)
    .agg(
        agr=('agr_id_key', 'nunique'),
        inn=('inn_key', 'nunique'),
        sum_calc=('commission_cft_calc', 'sum'),
        agr_nonzero=('has_cft_nonzero', 'sum'),
    )
    .sort_values('report_month')
)


## 4) Покрытие: источник → `final_df` и `final_df` → источник


In [ ]:
rows = []
presence_rows = []
detail_parts = []

for month in months:
    fin_m = fd3.loc[fd3['report_month'] == month]
    mpos_m = mpos_map.loc[mpos_map['report_month'] == month]
    cft_m = cft_map.loc[cft_map['report_month'] == month]
    cft_all_m = cft_map_all.loc[cft_map_all['report_month'] == month]

    fin_agr = set(fin_m['agr_id_key'].dropna())
    fin_inn = set(fin_m['inn_key'].dropna())
    mpos_agr = set(mpos_m['agr_id_key'].dropna())
    mpos_inn = set(mpos_m['inn_key'].dropna())
    cft_agr = set(cft_m['agr_id_key'].dropna())
    cft_inn = set(cft_m['inn_key'].dropna())
    mpos_agr_nz = set(mpos_m.loc[mpos_m['has_mpos_nonzero'], 'agr_id_key'].dropna())
    cft_agr_nz = set(cft_m.loc[cft_m['has_cft_nonzero'], 'agr_id_key'].dropna())

    specs = [
        ('agr_id', 'MPOS_RENT', mpos_agr, fin_agr),
        ('inn', 'MPOS_RENT', mpos_inn, fin_inn),
        ('agr_id', 'CFT_DOG_OPER', cft_agr, fin_agr),
        ('inn', 'CFT_DOG_OPER', cft_inn, fin_inn),
        ('agr_id', 'MPOS_RENT_nonzero', mpos_agr_nz, fin_agr),
        ('agr_id', 'CFT_DOG_OPER_nonzero', cft_agr_nz, fin_agr),
        ('agr_id', 'CFT_DOG_OPER_all_types', set(cft_all_m['agr_id_key'].dropna()), fin_agr),
    ]
    for grain, source, src_keys, fin_keys in specs:
        rec = coverage_block(src_keys, fin_keys)
        rec.update({'report_month': month, 'grain': grain, 'source': source})
        rows.append(rec)

    both = fin_agr & mpos_agr & cft_agr
    only_mpos = fin_agr & mpos_agr - cft_agr
    only_cft = fin_agr & cft_agr - mpos_agr
    neither = fin_agr - mpos_agr - cft_agr
    n_fin = len(fin_agr)
    presence_rows.append({
        'report_month': month,
        'n_final_agr': n_fin,
        'n_final_inn': len(fin_inn),
        'both': len(both),
        'only_mpos': len(only_mpos),
        'only_cft': len(only_cft),
        'neither': len(neither),
        'pct_both': round(100.0 * len(both) / n_fin, 2) if n_fin else np.nan,
        'pct_only_mpos': round(100.0 * len(only_mpos) / n_fin, 2) if n_fin else np.nan,
        'pct_only_cft': round(100.0 * len(only_cft) / n_fin, 2) if n_fin else np.nan,
        'pct_neither': round(100.0 * len(neither) / n_fin, 2) if n_fin else np.nan,
        'pct_has_mpos': round(100.0 * len(fin_agr & mpos_agr) / n_fin, 2) if n_fin else np.nan,
        'pct_has_cft': round(100.0 * len(fin_agr & cft_agr) / n_fin, 2) if n_fin else np.nan,
        'pct_has_any': round(100.0 * len((fin_agr & mpos_agr) | (fin_agr & cft_agr)) / n_fin, 2) if n_fin else np.nan,
    })

    det = fin_m[['report_month', 'inn_key', 'agr_id_key', 'commission_monthly_num']].drop_duplicates(
        ['report_month', 'agr_id_key']
    )
    det = det.merge(
        mpos_m[['report_month', 'agr_id_key', 'commission_mpos', 'has_mpos', 'has_mpos_nonzero']],
        on=['report_month', 'agr_id_key'],
        how='left',
    )
    det = det.merge(
        cft_m[['report_month', 'agr_id_key', 'commission_cft_calc', 'commission_cft_pay', 'has_cft', 'has_cft_nonzero']],
        on=['report_month', 'agr_id_key'],
        how='left',
    )
    det['has_mpos'] = det['has_mpos'].fillna(False)
    det['has_cft'] = det['has_cft'].fillna(False)
    det['presence'] = np.select(
        [
            det['has_mpos'] & det['has_cft'],
            det['has_mpos'] & ~det['has_cft'],
            ~det['has_mpos'] & det['has_cft'],
        ],
        ['both', 'only_mpos', 'only_cft'],
        default='neither',
    )
    detail_parts.append(det)

cov_df = pd.DataFrame(rows)
cov_df = cov_df[[
    'report_month', 'grain', 'source',
    'n_source', 'n_source_in_final', 'pct_source_in_final',
    'n_final', 'n_final_in_source', 'pct_final_has_source',
    'n_source_only', 'n_final_only',
]]
presence_df = pd.DataFrame(presence_rows)
detail_df = pd.concat(detail_parts, ignore_index=True) if detail_parts else pd.DataFrame()

print('=== Вхождение источника в final_df / покрытие final_df источником ===')
print('pct_source_in_final = доля ключей источника, которые есть в final_df')
print('pct_final_has_source = доля ключей final_df, у которых есть данные в источнике')
print('CFT scope:', cft_scope)
display(cov_df)

print('\\n=== Матрица наличия у договоров final_df ===')
display(presence_df)


## 5) Клиенты (ИНН) — та же матрица


In [ ]:
inn_presence_rows = []
for month in months:
    fin_m = fd3.loc[fd3['report_month'] == month]
    mpos_m = mpos_map.loc[mpos_map['report_month'] == month]
    cft_m = cft_map.loc[cft_map['report_month'] == month]
    fin_inn = set(fin_m['inn_key'].dropna())
    mpos_inn = set(mpos_m['inn_key'].dropna())
    cft_inn = set(cft_m['inn_key'].dropna())
    n_fin = len(fin_inn)
    both = fin_inn & mpos_inn & cft_inn
    only_mpos = fin_inn & mpos_inn - cft_inn
    only_cft = fin_inn & cft_inn - mpos_inn
    neither = fin_inn - mpos_inn - cft_inn
    inn_presence_rows.append({
        'report_month': month,
        'n_final_inn': n_fin,
        'both': len(both),
        'only_mpos': len(only_mpos),
        'only_cft': len(only_cft),
        'neither': len(neither),
        'pct_both': round(100.0 * len(both) / n_fin, 2) if n_fin else np.nan,
        'pct_only_mpos': round(100.0 * len(only_mpos) / n_fin, 2) if n_fin else np.nan,
        'pct_only_cft': round(100.0 * len(only_cft) / n_fin, 2) if n_fin else np.nan,
        'pct_neither': round(100.0 * len(neither) / n_fin, 2) if n_fin else np.nan,
        'pct_has_mpos': round(100.0 * len(fin_inn & mpos_inn) / n_fin, 2) if n_fin else np.nan,
        'pct_has_cft': round(100.0 * len(fin_inn & cft_inn) / n_fin, 2) if n_fin else np.nan,
        'pct_has_any': round(100.0 * len((fin_inn & mpos_inn) | (fin_inn & cft_inn)) / n_fin, 2) if n_fin else np.nan,
    })

inn_presence_df = pd.DataFrame(inn_presence_rows)
print('=== Матрица наличия у клиентов final_df (ИНН) ===')
display(inn_presence_df)


## 6) Вердикт + выгрузка


In [ ]:
def _fmt(v):
    if pd.isna(v):
        return 'n/a'
    return f'{v:.2f}%'


print('=== VERDICT ===')
print(f'Месяцы: {months}')
print(f'ЦФТ scope: {cft_scope} | tables: {cft_used}')
print()
for month in months:
    p = presence_df.loc[presence_df['report_month'] == month].iloc[0]
    ip = inn_presence_df.loc[inn_presence_df['report_month'] == month].iloc[0]
    agr_mpos = cov_df.loc[
        (cov_df['report_month'] == month) & (cov_df['grain'] == 'agr_id') & (cov_df['source'] == 'MPOS_RENT')
    ].iloc[0]
    agr_cft = cov_df.loc[
        (cov_df['report_month'] == month) & (cov_df['grain'] == 'agr_id') & (cov_df['source'] == 'CFT_DOG_OPER')
    ].iloc[0]
    inn_mpos = cov_df.loc[
        (cov_df['report_month'] == month) & (cov_df['grain'] == 'inn') & (cov_df['source'] == 'MPOS_RENT')
    ].iloc[0]
    inn_cft = cov_df.loc[
        (cov_df['report_month'] == month) & (cov_df['grain'] == 'inn') & (cov_df['source'] == 'CFT_DOG_OPER')
    ].iloc[0]
    print(f'--- {month} ---')
    print(
        f'  Договоры final_df: {int(p["n_final_agr"]):,} | '
        f'MPOS {_fmt(p["pct_has_mpos"])} | ЦФТ {_fmt(p["pct_has_cft"])} | '
        f'оба {_fmt(p["pct_both"])} | ни одного {_fmt(p["pct_neither"])}'
    )
    print(
        f'  Клиенты final_df: {int(ip["n_final_inn"]):,} | '
        f'MPOS {_fmt(ip["pct_has_mpos"])} | ЦФТ {_fmt(ip["pct_has_cft"])} | '
        f'оба {_fmt(ip["pct_both"])} | ни одного {_fmt(ip["pct_neither"])}'
    )
    print(
        f'  Вхождение источника в final_df (agr): '
        f'MPOS {int(agr_mpos["n_source_in_final"]):,}/{int(agr_mpos["n_source"]):,} = {_fmt(agr_mpos["pct_source_in_final"])} | '
        f'ЦФТ {int(agr_cft["n_source_in_final"]):,}/{int(agr_cft["n_source"]):,} = {_fmt(agr_cft["pct_source_in_final"])}'
    )
    print(
        f'  Вхождение источника в final_df (inn): '
        f'MPOS {int(inn_mpos["n_source_in_final"]):,}/{int(inn_mpos["n_source"]):,} = {_fmt(inn_mpos["pct_source_in_final"])} | '
        f'ЦФТ {int(inn_cft["n_source_in_final"]):,}/{int(inn_cft["n_source"]):,} = {_fmt(inn_cft["pct_source_in_final"])}'
    )

out_xlsx = OUT_DIR / f'coverage_{months[0]}_{months[-1]}.xlsx'
with pd.ExcelWriter(out_xlsx, engine='openpyxl') as w:
    cov_df.to_excel(w, sheet_name='coverage', index=False)
    presence_df.to_excel(w, sheet_name='final_agr_matrix', index=False)
    inn_presence_df.to_excel(w, sheet_name='final_inn_matrix', index=False)
    type_prof.to_excel(w, sheet_name='cft_vid_comiss', index=False)
    detail_df.to_excel(w, sheet_name='final_agr_detail', index=False)

print('\\nSaved:', out_xlsx)
print('Sheets: coverage | final_agr_matrix | final_inn_matrix | cft_vid_comiss | final_agr_detail')
